# 01 — Membangun Case Base
**Tujuan:** Baca semua PDF/TXT dari `data/raw/`, bersihkan teks, validasi, simpan hasil bersih.

In [6]:
import os, re, glob, logging, unicodedata
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

# ── PATH (relatif dari notebooks/) ──────────────────────────────
BASE        = Path('..').resolve()          # root projek-cbr-hukum/
INPUT_DIR   = BASE / 'data' / 'raw'         # ← letakkan PDF/TXT di sini
OUTPUT_DIR  = BASE / 'data' / 'raw' / 'cleaned'
LOG_DIR     = BASE / 'logs'

for d in [INPUT_DIR, OUTPUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Logger ───────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_DIR / 'cleaning.log', encoding='utf-8'),
    ]
)
log = logging.getLogger('step1')
print('Setup selesai ')

Setup selesai 


In [7]:
# ── Pola header/footer khas Direktori MA ────────────────────────
HEADER_PATTERNS = [
    r'direktori putusan mahkamah agung republik indonesia',
    r'mahkamah agung republik indonesia',
    r'putusan\.mahkamahagung\.go\.id',
    r'halaman\s*\d+\s*dari\s*\d+\s*halaman',
    r'catatan\s+catatan',
    r'-\s*\d+\s*-',
    r'={5,}', r'-{5,}', r'\f',
]

def extract_pdf(path: Path) -> str:
    try:
        from pdfminer.high_level import extract_text
        return extract_text(str(path)) or ''
    except Exception as e:
        log.error(f'Gagal ekstrak PDF {path.name}: {e}')
        return ''

def extract_txt(path: Path) -> str:
    try:
        return path.read_text(encoding='utf-8', errors='replace')
    except Exception as e:
        log.error(f'Gagal baca TXT {path.name}: {e}')
        return ''

def clean_text(raw: str) -> str:
    """Normalisasi unicode → lowercase → hapus header/footer → normalisasi spasi."""
    text = unicodedata.normalize('NFKC', raw).lower()
    for pat in HEADER_PATTERNS:
        text = re.sub(pat, ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]', ' ', text)  # karakter kontrol
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def validate(text: str) -> dict:
    """Validasi kualitas: min 500 karakter + 3 elemen kunci putusan."""
    has_putusan  = bool(re.search(r'putusan|mengadili', text))
    has_perkara  = bool(re.search(r'\d+/pid|\d+/pdt|nomor\s+\d+', text))
    has_dakwaan  = bool(re.search(r'terdakwa|dakwaan|penuntut', text))
    score = (has_putusan + has_perkara + has_dakwaan) / 3.0
    return {
        'valid'        : len(text) >= 500 and score >= 0.33,
        'char_count'   : len(text),
        'word_count'   : len(text.split()),
        'quality_score': round(score, 2),
    }

print('Fungsi siap ')

Fungsi siap 


In [8]:
import sys
!{sys.executable} -m pip install pdfminer.six

'c:\Users\Mukhamad' is not recognized as an internal or external command,
operable program or batch file.


In [9]:
# ── Jalankan Pipeline ────────────────────────────────────────────
files = sorted(list(INPUT_DIR.rglob('*.pdf')) + list(INPUT_DIR.rglob('*.txt')))
# Hapus file output yang sudah bernama case_XXX.txt agar tidak terbaca ulang
files = [f for f in files if not re.match(r'case_\d+\.txt', f.name)]

print(f'Ditemukan {len(files)} file input')

ok, skip = 0, 0
for idx, fpath in enumerate(tqdm(files, desc='Preprocessing'), start=1):
    case_id = f'case_{idx:03d}'
    raw     = extract_pdf(fpath) if fpath.suffix == '.pdf' else extract_txt(fpath)
    cleaned = clean_text(raw)
    val     = validate(cleaned)

    if not val['valid']:
        log.warning(f'SKIP {case_id} ({fpath.name}) | chars={val["char_count"]} | q={val["quality_score"]}')
        skip += 1
        continue

    out = OUTPUT_DIR / f'{case_id}.txt'
    out.write_text(cleaned, encoding='utf-8')
    log.info(f'OK   {case_id} | {val["word_count"]} kata | q={val["quality_score"]}')
    ok += 1

print(f'\n Selesai: {ok} berhasil, {skip} dilewati')
print(f' Output : {OUTPUT_DIR}')
if ok < 30:
    print(f'  Dokumen valid hanya {ok} — syarat minimal 30 belum terpenuhi!')

Ditemukan 60 file input


Preprocessing: 100%|██████████| 60/60 [00:32<00:00,  1.86it/s]


 Selesai: 60 berhasil, 0 dilewati
 Output : D:\Kuliah\Tugas Penalaran Komputer\Penalaran-Komputer_SubCpmk3\data\raw\cleaned


In [10]:
# ── Ringkasan ────────────────────────────────────────────────────
hasil = sorted(OUTPUT_DIR.glob('case_*.txt'))
print(f'Total file di data/raw/ : {len(hasil)}')
if hasil:
    sample = hasil[0].read_text(encoding='utf-8')
    print(f'\nPreview {hasil[0].name}:')
    print('-' * 60)
    print(sample[:400], '...')

Total file di data/raw/ : 60

Preview case_001.txt:
------------------------------------------------------------
disclaimer
kepaniteraan berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen mahkamah agung untuk pelayanan publik, transparansi dan akuntabilitas
pelaksanaan fungsi peradilan. namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari w ...
